# 因果发现 (Causal Discovery)

基于 LiNGAM 从观测数据学习变量间的有向无环图 (DAG)，识别对 PCE 的潜在因果驱动因素（区别于 SHAP 的关联性解释）。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.preprocessing import StandardScaler

import lingam

# 只使用 result.csv（你更新后的 dataset：包含更多指纹位，例如 Bit / FP2 Bit 等）
df = pd.read_csv('result.csv')

target_col = 'EQE (%)'
smiles_col = 'SMILES'

# 基本校验
if target_col not in df.columns:
    raise KeyError(f"Missing target column: {target_col}")

# 目标必须可转成数值
if not np.issubdtype(df[target_col].dtype, np.number):
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

# 自动识别特征列：除 SMILES 与 target 外的所有列；仅保留可转为数值的列
exclude = {target_col}
if smiles_col in df.columns:
    exclude.add(smiles_col)

feature_candidates = [c for c in df.columns if c not in exclude]

# 尽量保持 0/1 指纹；无法转数值的列会变 NaN 并被剔除
X_num = df[feature_candidates].apply(pd.to_numeric, errors='coerce')

# 删除全 NaN 的特征列
all_nan = X_num.columns[X_num.isna().all()].tolist()
X_num = X_num.drop(columns=all_nan)

# 缺失按 0 处理（对指纹位更合理；若是连续特征也可改成均值填充）
X_num = X_num.fillna(0.0)

feature_cols = X_num.columns.tolist()

print(f"Detected features: {len(feature_cols)} (dropped all-NaN: {len(all_nan)})")

# 保留 SMILES/target 方便追溯（SMILES 不参与建模）
keep_cols = []
if smiles_col in df.columns:
    keep_cols.append(smiles_col)
keep_cols += [target_col]

df = df[keep_cols].copy()
df[target_col] = df[target_col].astype(float)

df.head()

In [ ]:
# 选择用于因果发现的变量：自动识别的指纹/数值特征位为输入；目标为 EQE (%)
# 注意：SMILES 不作为输入特征（只用于追溯）

# 构造矩阵：最后一列为目标（供 LiNGAM 使用）
# X_num / feature_cols 来自上一个单元
X = X_num[feature_cols].copy()
y = df[target_col].copy()

Z = pd.concat([X, y.rename(target_col)], axis=1)

# -------- 清洗：删除含 NaN/inf 的列 + std==0 的列（避免 LiNGAM 内部除以 0） --------
Z = Z.replace([np.inf, -np.inf], np.nan)

# 目标列必须有效
if Z[target_col].isna().any():
    bad = int(Z[target_col].isna().sum())
    raise ValueError(f"Target {target_col} has NaN (n={bad}) after coercion.")
if float(Z[target_col].std(ddof=0)) == 0.0:
    raise ValueError(f"Target {target_col} has zero std; cannot learn causal structure.")

# 只对特征做列筛除（保留目标列）
feature_cols = [c for c in Z.columns if c != target_col]

nan_cols = [c for c in feature_cols if Z[c].isna().any()]
Z = Z.drop(columns=nan_cols)

feature_cols = [c for c in Z.columns if c != target_col]
zero_std_cols = [c for c in feature_cols if float(Z[c].std(ddof=0)) == 0.0]
Z = Z.drop(columns=zero_std_cols)

feature_cols = [c for c in Z.columns if c != target_col]

print(f"Dropped {len(nan_cols)} NaN/inf columns and {len(zero_std_cols)} zero-std columns.")
print(f"Remaining features (before filtering): {len(feature_cols)}")

# -------- 为 LiNGAM 做降维/去共线（避免维度过高导致不稳定/耗时） --------
# 1) 先按与目标的 |相关性| 取 Top-K
# 重要：必须保证 #features < #samples（否则 DirectLiNGAM 内部 LassoLarsIC 会报错）
import os

# nbconvert 后台跑：适当降配，减少 kernel 因内存/数值问题崩溃的概率
_IS_NBCONVERT = os.environ.get('NBCONVERT_RUN', '').strip() == '1'
USER_MAX_FEATURES = 150 if _IS_NBCONVERT else 300

n_samples = int(Z.shape[0])
# 留出余量，避免边界情况
safe_cap = max(5, n_samples - 2)
LINGAM_MAX_FEATURES = min(USER_MAX_FEATURES, safe_cap)
print(f"LiNGAM feature cap: min({USER_MAX_FEATURES}, n_samples-2={safe_cap}) => {LINGAM_MAX_FEATURES}")

if LINGAM_MAX_FEATURES is not None and len(feature_cols) > LINGAM_MAX_FEATURES:
    yv = Z[target_col]
    corrs = Z[feature_cols].corrwith(yv).abs()
    corrs = corrs.replace([np.inf, -np.inf], np.nan).dropna()
    feature_cols = corrs.sort_values(ascending=False).head(LINGAM_MAX_FEATURES).index.tolist()
    Z = Z[feature_cols + [target_col]]
    print(f"LINGAM: kept top {len(feature_cols)} features by |corr| with {target_col}.")

# 2) 再做一次去共线（删除与已选特征高度相关的冗余列）
CORR_DROP_THRESHOLD = 0.999
if len(feature_cols) > 1:
    cmat = Z[feature_cols].corr().abs()
    upper = cmat.where(np.triu(np.ones(cmat.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > CORR_DROP_THRESHOLD)]
    if to_drop:
        Z = Z.drop(columns=to_drop)
        feature_cols = [c for c in Z.columns if c != target_col]
        print(f"LINGAM: dropped {len(to_drop)} highly-collinear features (>|r|={CORR_DROP_THRESHOLD}).")

cols = feature_cols + [target_col]
print(f"Remaining features (final): {len(feature_cols)}")

# 标准化（LiNGAM 对尺度敏感）
scaler = StandardScaler()
Z_scaled = scaler.fit_transform(Z[cols])
Z_df = pd.DataFrame(Z_scaled, columns=cols)
Z_df.head()

In [ ]:
# ── 1. Shapiro-Wilk 非高斯性检验 ──────────────────────────────────────────────
# 目的：验证 LiNGAM 的核心识别性假设（非高斯噪声）是否成立

from scipy import stats
import warnings
import numpy as np
import pandas as pd

SW_ALPHA = 0.05          # 显著性水平；p < alpha → 拒绝正态 → 支持 LiNGAM 假设
SW_MAX_N = 5000          # Shapiro-Wilk 官方限制：n > 5000 时精度下降，自动切换 K²
SW_SAMPLE = 5000         # 超限时的抽样量
sw_results = []

for col in Z_df.columns:
    x = Z_df[col].dropna().values
    if len(x) > SW_MAX_N:
        # 样本过大时用 D'Agostino-Pearson K² 检验（适用大样本）
        rng_sw = np.random.default_rng(42)
        x_sub = rng_sw.choice(x, size=SW_SAMPLE, replace=False)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            stat, p = stats.shapiro(x_sub)
        test_name = f"Shapiro-Wilk (n={SW_SAMPLE} subsample)"
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            stat, p = stats.shapiro(x)
        test_name = "Shapiro-Wilk"
        
    # 偏度 & 峰度：进一步量化非高斯程度
    skew  = float(stats.skew(x))
    kurt  = float(stats.kurtosis(x))   # excess kurtosis，正态 = 0
    sw_results.append({
        'variable':  col,
        'test':      test_name,
        'W_stat':    round(stat, 6),
        'p_value':   p,
        'reject_normal': p < SW_ALPHA,   # True → 非高斯 → 支持 LiNGAM
        'skewness':  round(skew, 4),
        'excess_kurtosis': round(kurt, 4),
        'n':         len(x),
    })

sw_df = pd.DataFrame(sw_results).sort_values('p_value', ascending=True)

print("=" * 60)
print("Shapiro-Wilk 非高斯性检验汇总")
print("=" * 60)
n_total = len(sw_df)
n_nongaussian = sw_df['reject_normal'].sum()
pct_nongaussian = n_nongaussian / n_total * 100
print(f"检验变量总数    : {n_total}")
print(f"拒绝正态（非高斯）: {n_nongaussian}  ({pct_nongaussian:.1f}%)")
print(f"\n结论：{'✓ 绝大多数变量非高斯，LiNGAM 方向识别假设成立' if pct_nongaussian > 80 else '⚠ 部分变量近似正态，结果需审慎解读'}")
print()


# ── 2. 线性 vs 单调非线性关系排查 ──────────────────────────────────────────────
# 目的：对比 Pearson(线性) 和 Spearman(单调非线性)，排查是否存在强非线性关系
# 逻辑：如果两者绝对值差异很大，说明存在 LiNGAM 的纯线性假设难以捕捉的非线性关系

NONLINEAR_THRESHOLD = 0.15  # 差异阈值：Pearson和Spearman相差 > 0.15 视为存在较强非线性

corr_results = []
feature_cols_only = [c for c in Z_df.columns if c != target_col]

for col in feature_cols_only:
    # 计算 Pearson (线性相关)
    pearson_r, _ = stats.pearsonr(Z_df[col], Z_df[target_col])
    # 计算 Spearman (单调相关，不局限于直线)
    spearman_rho, _ = stats.spearmanr(Z_df[col], Z_df[target_col])
    
    # 计算绝对值差异
    diff = abs(abs(pearson_r) - abs(spearman_rho))
    
    corr_results.append({
        'variable': col,
        'Pearson_r (Linear)': round(pearson_r, 4),
        'Spearman_rho (Monotonic)': round(spearman_rho, 4),
        'Abs_Diff': round(diff, 4),
        'Strong_Nonlinear_Risk': diff > NONLINEAR_THRESHOLD
    })

corr_df = pd.DataFrame(corr_results).sort_values('Abs_Diff', ascending=False)

n_nonlinear_risk = corr_df['Strong_Nonlinear_Risk'].sum()
print("=" * 60)
print("线性 vs 单调非线性 排查汇总 (相对于目标变量)")
print("=" * 60)
print(f"存在强非线性风险的特征数量 (差异 > {NONLINEAR_THRESHOLD}): {n_nonlinear_risk} / {len(feature_cols_only)}")

if n_nonlinear_risk > 0:
    print(f"\n⚠ 发现部分特征表现出较强的非线性单调关系。LiNGAM 会提取它们的一阶线性近似，但可能低估其总效应。")
    print("--- 差异最大的 Top 特征 (非线性风险最高) ---")
    display(corr_df.head(n_nonlinear_risk if n_nonlinear_risk <= 10 else 10))
else:
    print(f"\n✓ 绝大多数特征的 Pearson 和 Spearman 表现一致。这证明了特征与目标之间的关系以线性为主。")
    print("✓ LiNGAM 的线性假设在本数据集中高度合理！")

# 保存结果供汇报使用
_out = 'result_eqe_linearity_check_pearson_vs_spearman.csv'
corr_df.to_csv(_out, index=False)
print(f"\n已保存: {_out}")

In [ ]:
# （移除重复的 Shapiro-Wilk 单元格；上一个单元格已包含并会输出 CSV）


In [ ]:
import time
t0 = time.perf_counter()
model = lingam.DirectLiNGAM()
model.fit(Z_df)  # 和你正式用的一样
print("single fit seconds:", time.perf_counter() - t0)

In [ ]:
# # DirectLiNGAM：线性非高斯无环模型，从数据中学习 DAG
# model = lingam.DirectLiNGAM()
# model.fit(Z_df)

# # 邻接矩阵 B[i,j] 表示 j -> i 的边权（即 i 受 j 的线性影响）
B = model.adjacency_matrix_
print('Causal order (从原因到结果):', model.causal_order_)

In [ ]:
# 构建有向图：仅保留边权绝对值大于阈值的边，得到稀疏 DAG
# 需先按顺序运行：加载数据 → 构造 Z/cols → DirectLiNGAM 得到 B（否则会出现 NameError）
if 'cols' not in globals() or 'B' not in globals():
    raise RuntimeError('请先运行上方单元格：构造 Z_df/cols，并完成 model.fit 得到 B。')

threshold = 0.05
n = len(cols)
G = nx.DiGraph()
G.add_nodes_from(cols)

for i in range(n):
    for j in range(n):
        if i != j and abs(B[i, j]) > threshold:
            G.add_edge(cols[j], cols[i], weight=float(B[i, j]))

# 直接指向目标变量的变量（潜在因果驱动）
parents_of_target = list(G.predecessors(target_col))
print(f'直接指向 {target_col} 的变量（潜在因果驱动）:', parents_of_target)
if parents_of_target:
    for u in parents_of_target:
        w = G.edges[u, target_col]['weight']
        print(f'  {u} -> {target_col}: 权重 {w:.4f}')

In [ ]:
# 可视化：仅显示与目标变量相连的边 + 这些节点的内部边，避免图过密
nodes_to_show = set(parents_of_target) | {target_col}
for u in list(nodes_to_show):
    nodes_to_show |= set(G.predecessors(u))
sub = G.subgraph(nodes_to_show)

pos = nx.spring_layout(sub, k=1.5, seed=42)
plt.figure(figsize=(12, 8))
nx.draw_networkx_nodes(sub, pos, node_color='lightblue', node_size=800)
nx.draw_networkx_nodes(sub, pos, nodelist=[target_col], node_color='orange', node_size=1200)
nx.draw_networkx_labels(sub, pos, font_size=8)
nx.draw_networkx_edges(sub, pos, edge_color='gray', arrows=True, arrowsize=20)
plt.title(f'Causal DAG (LiNGAM): Subgraph related to {target_col}')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 可选：保存邻接矩阵与“目标变量的直接原因”列表，供后续分析
adj_df = pd.DataFrame(B, index=cols, columns=cols)

adj_out = 'result_eqe_causal_adjacency_lingam.csv'
parents_out = 'result_eqe_causal_parents_of_target.csv'
edges_out = 'result_eqe_causal_edges_thresholded.csv'

adj_df.to_csv(adj_out)
pd.Series(parents_of_target).to_csv(parents_out, index=False, header=['parent'])

# 同步保存阈值化后的边表（便于排序/筛选）
edge_rows = [(u, v, float(d.get('weight', np.nan))) for u, v, d in G.edges(data=True)]
edges_df = pd.DataFrame(edge_rows, columns=['cause', 'effect', 'weight']).sort_values('weight', key=lambda s: s.abs(), ascending=False)
edges_df.to_csv(edges_out, index=False)

print(f'已保存: {adj_out}, {parents_out}, {edges_out}')

（已按需求精简：仅保留基于 `result.csv` 的 LiNGAM 拟合与结果导出；其余旧流程已移除。）

In [ ]:
# ── 3) Bootstrap 稳定性测试 + Top20 特征筛选 ─────────────────────────────
# 目标：评估 direct edges (feature -> EQE) 的稳定性，并输出影响最大的前 20 个特征

import numpy as np
import pandas as pd

# 使用主模型学到的阈值化边表（edges_df / threshold 来自前面单元格）
if 'B' not in globals() or 'cols' not in globals() or 'threshold' not in globals():
    raise RuntimeError('请先运行到 LiNGAM 拟合得到 B/cols，并完成构图单元格中的 threshold。')

TARGET = target_col

# 1) Top20：用邻接矩阵 B 中「特征 -> 目标」的完整直接系数排序（不依赖 |B|>threshold 的稀疏边）。
# 若仅用 edges_df（阈值化后），指向 EQE 的边可能远少于 20，CSV 行数会不足 20。
#
# 重要：B[i,j] 是 LiNGAM 在因果顺序下估计的「直接」效应（对 EQE 还控制了图中排在 EQE 之前的其它变量）。
# 这与「边际」Pearson 相关 marginal_corr 不是同一回事：直接系数为负时，边际相关仍常为正（混淆/链式结构）。
# 因此不能只看 weight 列就说「全是负面影响」——请对照 marginal_corr。
t_idx = cols.index(TARGET)
direct_rows = []
for j, cname in enumerate(cols):
    if cname == TARGET:
        continue
    w = float(B[t_idx, j])
    direct_rows.append({'cause': cname, 'effect': TARGET, 'weight': w})
direct_full = pd.DataFrame(direct_rows)
direct_full['abs_weight'] = direct_full['weight'].abs()

# 边际 Pearson 相关（与 Z_df 一致；有正负号，便于与直接系数对比）
yv = Z_df[TARGET]
direct_full['marginal_corr'] = direct_full['cause'].apply(lambda c: float(Z_df[c].corr(yv)))
direct_full['abs_marginal_corr'] = direct_full['marginal_corr'].abs()

# 次要排序键：|边际相关|（用于 LiNGAM 直接系数为 0 或并列时补足到 20 行）
direct_full['abs_corr_with_target'] = direct_full['abs_marginal_corr']

# 正向 Top20（仅 w>0 的直接系数；DAG 里正系数可能不足 20，行数可少于 20）
top20_pos = (
    direct_full[direct_full['weight'] > 0]
    .sort_values('weight', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# |B| Top20：主键 |LiNGAM 直接系数|，次键 |corr|，保证在特征>=20 时输出 20 行有意义排序
top20_abs = (
    direct_full
    .sort_values(['abs_weight', 'abs_corr_with_target', 'weight'], ascending=[False, False, False])
    .head(20)
    .reset_index(drop=True)
)

out_pos = 'result_eqe_top20_features_positive_direct_to_target.csv'
out_abs = 'result_eqe_top20_features_abs_direct_to_target.csv'

top20_pos[['cause', 'effect', 'weight', 'marginal_corr']].to_csv(out_pos, index=False)
top20_abs[['cause', 'effect', 'weight', 'abs_weight', 'marginal_corr', 'abs_marginal_corr']].to_csv(out_abs, index=False)

print('Saved:', out_pos)
print('Saved:', out_abs)
n_pos = int((direct_full['weight'] > 0).sum())
n_pos_marg = int((direct_full['marginal_corr'] > 0).sum())
print(f"Top20 |w|: {len(top20_abs)} 行; Top20 w>0: {len(top20_pos)} 行（直接系数>0 的特征共 {n_pos} 个）")
print(f"边际相关>0 的特征数: {n_pos_marg} / {len(direct_full)}（与「直接系数」符号不必一致）")
print(f"Top20 |直接系数| 中 marginal_corr>0 的行数: {int((top20_abs['marginal_corr'] > 0).sum())}")

# 2) Bootstrap 稳定性（直接边 feature -> target）
# 在标准化后的 Z_df 上做 bootstrap；每次拟合 DirectLiNGAM，记录指向 target 的边权

# 注意：nbconvert 后台执行更容易因资源限制/数值不稳定导致 kernel 死亡。
# 因此在后台执行时自动降配（更少 bootstrap 次数）；交互运行时保持更高次数。
import os
_IS_NBCONVERT = os.environ.get('NBCONVERT_RUN', '').strip() == '1'
N_BOOT = 15 if _IS_NBCONVERT else 50
RANDOM_SEED = 42
THRESH = float(threshold)

print(f"Starting bootstrap: N_BOOT={N_BOOT}, threshold={THRESH}", flush=True)

rng = np.random.default_rng(RANDOM_SEED)
Z_np = Z_df.to_numpy(dtype=float)
col_names = list(Z_df.columns)

t_idx = col_names.index(TARGET)
feature_names = [c for c in col_names if c != TARGET]

# 收集每次 bootstrap 的 direct weights（小于阈值记为 0）
W_boot = {f: [] for f in feature_names}

for b in range(N_BOOT):
    idx = rng.integers(0, Z_np.shape[0], size=Z_np.shape[0])
    Zb = Z_np[idx, :]
    m = lingam.DirectLiNGAM()
    m.fit(Zb)
    Bb = m.adjacency_matrix_.copy()  # B[i,j]: j -> i

    # direct weights into target
    for j, fname in enumerate(col_names):
        if fname == TARGET:
            continue
        w = float(Bb[t_idx, j])
        if abs(w) < THRESH:
            w = 0.0
        W_boot[fname].append(w)

    if (b + 1) % 10 == 0:
        print(f"bootstrap {b+1}/{N_BOOT} done", flush=True)

rows = []
for f in feature_names:
    arr = np.asarray(W_boot[f], dtype=float)
    present = arr != 0
    freq = float(present.mean())
    mean_w = float(arr.mean())
    std_w = float(arr.std(ddof=0))
    pos_ratio = float((arr[present] > 0).mean()) if present.any() else np.nan
    rows.append({
        'feature': f,
        'freq_nonzero(|w|>=threshold)': freq,
        'mean_weight': mean_w,
        'std_weight': std_w,
        'pos_ratio_given_present': pos_ratio,
        'threshold': THRESH,
        'n_boot': N_BOOT,
    })

boot_df = pd.DataFrame(rows)
boot_df['abs_mean_weight'] = boot_df['mean_weight'].abs()
boot_df = boot_df.sort_values(['abs_mean_weight', 'freq_nonzero(|w|>=threshold)'], ascending=False)

boot_out = 'result_eqe_bootstrap_direct_edges_to_target.csv'
boot_df.to_csv(boot_out, index=False)
print('Saved:', boot_out)

# 同时输出：bootstrap 意义下的 Top20（正向/绝对值）
boot_top20_pos = boot_df[boot_df['mean_weight'] > 0].head(20)
boot_top20_abs = boot_df.head(20)

boot_top20_pos.to_csv('result_eqe_bootstrap_top20_positive_direct_to_target.csv', index=False)
boot_top20_abs.to_csv('result_eqe_bootstrap_top20_abs_direct_to_target.csv', index=False)
print('Saved: result_eqe_bootstrap_top20_positive_direct_to_target.csv')
print('Saved: result_eqe_bootstrap_top20_abs_direct_to_target.csv')

In [ ]:
# ── 4) 结果可视化（保存 PNG）────────────────────────────────────────────
# 说明：尽量使用前面单元格已经生成的对象（B/cols/adj_df/edges_df/direct_full/top20_abs/boot_df 等）。

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 兼容：如果 adj_df / edges_df 没在内存里（比如只运行到某一段），就从 CSV 读回
if 'adj_df' not in globals():
    if os.path.exists('result_eqe_causal_adjacency_lingam.csv'):
        adj_df = pd.read_csv('result_eqe_causal_adjacency_lingam.csv', index_col=0)

if 'edges_df' not in globals():
    if os.path.exists('result_eqe_causal_edges_thresholded.csv'):
        edges_df = pd.read_csv('result_eqe_causal_edges_thresholded.csv')

if 'direct_full' not in globals():
    if 'B' in globals() and 'cols' in globals() and 'Z_df' in globals():
        t_idx = cols.index(target_col)
        rows = []
        yv = Z_df[target_col]
        for j, cname in enumerate(cols):
            if cname == target_col:
                continue
            w = float(B[t_idx, j])
            rows.append({
                'cause': cname,
                'effect': target_col,
                'weight': w,
                'abs_weight': abs(w),
                'marginal_corr': float(Z_df[cname].corr(yv)),
            })
        direct_full = pd.DataFrame(rows)
        direct_full['abs_marginal_corr'] = direct_full['marginal_corr'].abs()

if 'boot_df' not in globals() and os.path.exists('result_eqe_bootstrap_direct_edges_to_target.csv'):
    boot_df = pd.read_csv('result_eqe_bootstrap_direct_edges_to_target.csv')

os.makedirs('.', exist_ok=True)

# 4.1 邻接矩阵热力图（只画目标相关的 TopN 节点，避免过大）
if 'adj_df' in globals() and adj_df is not None:
    TARGET = target_col
    if TARGET in adj_df.index:
        # 选取：对目标的直接边 |B| TopN + 目标本身
        N = min(40, adj_df.shape[0] - 1)
        col_abs = adj_df.loc[TARGET].drop(index=TARGET).abs().sort_values(ascending=False)
        top_nodes = col_abs.head(N).index.tolist()
        nodes = top_nodes + [TARGET]

        sub_adj = adj_df.loc[nodes, nodes]
        vmax = float(np.nanpercentile(np.abs(sub_adj.values), 98)) if sub_adj.size else 1.0
        vmax = max(vmax, 1e-6)

        plt.figure(figsize=(10, 8))
        plt.imshow(sub_adj.values, cmap='coolwarm', vmin=-vmax, vmax=vmax, aspect='auto')
        plt.colorbar(label='B[i,j] (j → i)')
        plt.xticks(range(len(nodes)), nodes, rotation=90, fontsize=7)
        plt.yticks(range(len(nodes)), nodes, fontsize=7)
        plt.title(f'LiNGAM adjacency heatmap (Top{N} causes to {TARGET})')
        plt.tight_layout()
        plt.savefig('result_eqe_adjacency_heatmap_topN.png', dpi=200)
        plt.show()

# 4.2 指向目标的直接系数 TopK 条形图（正/负分色）
if 'direct_full' in globals() and direct_full is not None and len(direct_full) > 0:
    dfk = direct_full.copy()
    dfk = dfk.sort_values('abs_weight', ascending=False).head(30).iloc[::-1]
    colors = ['tab:orange' if w > 0 else 'tab:blue' for w in dfk['weight'].values]

    plt.figure(figsize=(10, 8))
    plt.barh(dfk['cause'].values, dfk['weight'].values, color=colors)
    plt.axvline(0, color='k', linewidth=1)
    plt.title(f'Top30 direct effects to {target_col} (LiNGAM B[target, feature])')
    plt.xlabel('direct weight')
    plt.tight_layout()
    plt.savefig('result_eqe_top30_direct_effects_bar.png', dpi=200)
    plt.show()

    # 4.3 直接系数 vs 边际相关（看符号不一致/混淆）
    plt.figure(figsize=(7, 6))
    x = dfk['marginal_corr'].values if 'marginal_corr' in dfk.columns else direct_full['marginal_corr'].values
    y = dfk['weight'].values
    plt.scatter(x, y, s=40, alpha=0.8)
    plt.axhline(0, color='k', linewidth=1)
    plt.axvline(0, color='k', linewidth=1)
    plt.xlabel('marginal Pearson corr with target')
    plt.ylabel('direct LiNGAM weight to target')
    plt.title('Direct effect vs marginal correlation (sign may differ)')
    plt.tight_layout()
    plt.savefig('result_eqe_direct_vs_marginal_scatter.png', dpi=200)
    plt.show()

# 4.4 Bootstrap 稳定性散点图：出现频率 vs |均值效应|（越右上越稳定且强）
if 'boot_df' in globals() and boot_df is not None and len(boot_df) > 0:
    bdf = boot_df.copy()
    if 'abs_mean_weight' not in bdf.columns:
        bdf['abs_mean_weight'] = bdf['mean_weight'].abs()

    # 取最显著的前若干个，避免点太密
    topn = min(200, len(bdf))
    bplot = bdf.sort_values('abs_mean_weight', ascending=False).head(topn)

    plt.figure(figsize=(8, 6))
    plt.scatter(
        bplot['freq_nonzero(|w|>=threshold)'].values,
        bplot['abs_mean_weight'].values,
        s=35,
        alpha=0.75,
    )
    plt.xlabel('bootstrap freq(|w|>=threshold)')
    plt.ylabel('|mean direct weight|')
    plt.title(f'Bootstrap stability to {target_col} (top {topn} by |mean|)')
    plt.tight_layout()
    plt.savefig('result_eqe_bootstrap_stability_scatter.png', dpi=200)
    plt.show()

    # 4.5 最稳定 Top20（按 |mean| 排）条形图
    top20 = bdf.sort_values('abs_mean_weight', ascending=False).head(20).iloc[::-1]
    colors = ['tab:orange' if w > 0 else 'tab:blue' for w in top20['mean_weight'].values]
    plt.figure(figsize=(10, 7))
    plt.barh(top20['feature'].values, top20['mean_weight'].values, color=colors)
    plt.axvline(0, color='k', linewidth=1)
    plt.xlabel('bootstrap mean weight (thresholded)')
    plt.title(f'Bootstrap Top20 direct effects to {target_col}')
    plt.tight_layout()
    plt.savefig('result_eqe_bootstrap_top20_bar.png', dpi=200)
    plt.show()

print('可视化已保存：')
for fn in [
    'result_eqe_adjacency_heatmap_topN.png',
    'result_eqe_top30_direct_effects_bar.png',
    'result_eqe_direct_vs_marginal_scatter.png',
    'result_eqe_bootstrap_stability_scatter.png',
    'result_eqe_bootstrap_top20_bar.png',
]:
    if os.path.exists(fn):
        print(' -', fn)


In [ ]:
# ── 5) 给 Origin 用的“模型效果好”可视化数据导出 ─────────────────────────
# 思路：LiNGAM 学到的是结构方程 x_i = Σ_j B[i,j] x_j + e_i。
# 用它对目标变量做“结构方程重构”，如果 R² 高、残差较小且呈非高斯（与假设一致），可作为“拟合质量/解释力”的展示。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

if 'B' not in globals() or 'cols' not in globals() or 'Z_df' not in globals():
    raise RuntimeError('请先运行到 DirectLiNGAM 拟合（得到 B/cols）以及 Z_df 构造。')

TARGET = target_col
if TARGET not in cols:
    raise RuntimeError(f'TARGET {TARGET} not in cols')

t_idx = cols.index(TARGET)
Xmat = Z_df[cols].to_numpy(dtype=float)

# 结构方程预测：y_hat = Σ_j B[target,j] * x_j
w = np.asarray(B[t_idx, :], dtype=float)
# 自回归项置零（理论上应为 0）
w[t_idx] = 0.0

y_true = Xmat[:, t_idx]
y_pred = Xmat @ w
resid = y_true - y_pred

r2 = float(r2_score(y_true, y_pred))
mae = float(mean_absolute_error(y_true, y_pred))
rmse = float(mean_squared_error(y_true, y_pred, squared=False))

print(f"LiNGAM structural reconstruction for {TARGET} (standardized scale)")
print(f"R^2  : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

# 导出给 Origin：散点（真实 vs 预测）+ 残差
origin_fit = pd.DataFrame({
    'true_target_std': y_true,
    'pred_target_std': y_pred,
    'residual_std': resid,
})
origin_fit.to_csv('origin_eqe_structural_fit_scatter.csv', index=False)
print('Saved: origin_eqe_structural_fit_scatter.csv')

# 也导出每个变量的结构方程 R²（可在 Origin 做条形图/点图）
r2_rows = []
for i, name in enumerate(cols):
    wi = np.asarray(B[i, :], dtype=float)
    wi[i] = 0.0
    yi = Xmat[:, i]
    yhat_i = Xmat @ wi
    ri = float(r2_score(yi, yhat_i))
    r2_rows.append({'variable': name, 'structural_R2': ri})

r2_df = pd.DataFrame(r2_rows).sort_values('structural_R2', ascending=False)
r2_df.to_csv('origin_structural_R2_all_variables.csv', index=False)
print('Saved: origin_structural_R2_all_variables.csv')

# ── matplotlib 快速图（你也可以只用 Origin 画得更漂亮） ──
# 1) True vs Pred
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, s=22, alpha=0.75)
mn = float(min(y_true.min(), y_pred.min()))
mx = float(max(y_true.max(), y_pred.max()))
plt.plot([mn, mx], [mn, mx], 'k--', linewidth=1)
plt.xlabel('True target (std)')
plt.ylabel('Pred target (std)')
plt.title(f'{TARGET}: structural fit (R²={r2:.3f})')
plt.tight_layout()
plt.savefig('origin_like_eqe_true_vs_pred.png', dpi=220)
plt.show()

# 2) Residual histogram
plt.figure(figsize=(6, 4))
plt.hist(resid, bins=30, alpha=0.85, color='tab:gray')
plt.xlabel('Residual (std)')
plt.ylabel('Count')
plt.title(f'{TARGET}: residual distribution')
plt.tight_layout()
plt.savefig('origin_like_eqe_residual_hist.png', dpi=220)
plt.show()

# 3) QQ plot（残差是否近似正态；通常会偏离，反而可支持“非高斯噪声”假设）
try:
    import scipy.stats as st

    plt.figure(figsize=(5.5, 5.5))
    st.probplot(resid, dist='norm', plot=plt)
    plt.title(f'{TARGET}: residual QQ plot')
    plt.tight_layout()
    plt.savefig('origin_like_eqe_residual_qq.png', dpi=220)
    plt.show()
except Exception as e:
    print('QQ plot skipped:', repr(e))

print('Saved PNGs: origin_like_eqe_true_vs_pred.png, origin_like_eqe_residual_hist.png, origin_like_eqe_residual_qq.png')
